# Isolation Forest Anomaly Detection for TLS Profiling

This notebook evaluates one-class Isolation Forest baselines on extracted TLS traffic features. The dataset paths are parameterized so the same experiment can be run on one or more parquet partitions, and the target categories are controlled through `category_labels`. For each target label, the detector is trained only on flows from that label and then evaluated on its ability to separate that label from the rest of the traffic.


In [1]:
import sys
from pathlib import Path
sys.path.append('../../src')

### PARAMETERS:

In [2]:
train_features_path = ["../data-ext/malware_train.parquet", "../data-ext/apps_train.parquet"]
val_features_path = ["../data-ext/malware_val.parquet", "../data-ext/apps_val.parquet"]
test_features_path = ["../data-ext/malware_test.parquet", "../data-ext/apps_test.parquet"]
train_labels_path = ["../data-ext/malware_train_labels.parquet", "../data-ext/apps_train_labels.parquet"]
val_labels_path = ["../data-ext/malware_val_labels.parquet", "../data-ext/apps_val_labels.parquet"]
test_labels_path = ["../data-ext/malware_test_labels.parquet", "../data-ext/apps_test_labels.parquet"]

category_labels = ["system", "malware", "application"]


## Feature Groups

- `FLOW`: basic bidirectional flow statistics such as bytes, packets, rates, and duration (`bs`, `ps`, `br`, `pr`, `td`)
- `CTLS`: client-side TLS metadata (`tls.cver`, `tls.ccs`, `tls.cext`, `tls.csg`, `tls.alpn`, `tls.csv`)
- `STLS`: server-side TLS metadata (`tls.sver`, `tls.scs`, `tls.sext`, `tls.ssv`)
- `REC`: ordered sequence of signed TLS record lengths (`tls.rec`, first 20 records)

The experiments below sweep over every non-empty combination of these groups.


In [3]:
feature_groups = {
    "FLOW": ["bs", "ps", "br", "pr", "td"],
    "CTLS": ["tls.cver", "tls.ccs", "tls.cext", "tls.csg", "tls.alpn", "tls.csv"],
    "STLS": ["tls.sver", "tls.scs", "tls.sext", "tls.ssv"],
    "REC": ["tls.rec"],
}

FLOW = feature_groups["FLOW"]
CTLS = feature_groups["CTLS"]
STLS = feature_groups["STLS"]
REC = feature_groups["REC"]

In [4]:
import pandas as pd
from tls_profiling.preprocessing import build_and_fit_preprocessor

def read_parquet_files(paths):
    if isinstance(paths, (str, Path)):
        paths = [paths]
    return pd.concat([pd.read_parquet(Path(path)) for path in paths], ignore_index=True)

print("Loading extracted features from parameterized parquet paths.")
df_train = read_parquet_files(train_features_path)
df_val = read_parquet_files(val_features_path)
df_test = read_parquet_files(test_features_path)
df_train_label = read_parquet_files(train_labels_path)
df_val_label = read_parquet_files(val_labels_path)
df_test_label = read_parquet_files(test_labels_path)

preprocessor = build_and_fit_preprocessor(df_train)
print("Built preprocessor from df_train.")


Loading extracted features from parameterized parquet paths.
Built preprocessor from df_train.


In [5]:
from tls_profiling.baselines.model_isolation_forest import IsolationForestDetector, Config
import numpy as np

def train_detector(train: np.ndarray) -> IsolationForestDetector:
    detector = IsolationForestDetector(Config())
    detector.fit(train)
    return detector

from sklearn.metrics import PrecisionRecallDisplay, RocCurveDisplay
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_recall_curve

import warnings

warnings.filterwarnings(
    "ignore",
    message=r"unknown class\(es\) .* will be ignored",
    category=UserWarning,
    module=r"sklearn\.preprocessing\._label",
)

def select_feature_set(x, feature_set):
    if not feature_set:
        return x

    selected_columns = [
        column for column in x.columns
        if any(column.startswith(prefix) for prefix in feature_set)
    ]
    return x.loc[:, selected_columns]

def choose_f1_threshold(y_true, anomaly_score):
    precision, recall, thresholds = precision_recall_curve(y_true, anomaly_score)

    if len(thresholds) == 0:
        return float("inf")

    f1_scores = (2 * precision[:-1] * recall[:-1]) / np.clip(
        precision[:-1] + recall[:-1],
        a_min=1e-12,
        a_max=None,
    )
    best_idx = int(np.nanargmax(f1_scores))
    return float(thresholds[best_idx])

def evaluate(feature_set, normal_label):
    # x_train: only normal traffic
    x_train = df_train.loc[df_train_label["category"] == normal_label].reset_index(drop=True)
    # x_val: mixed traffic used to tune the F1 threshold
    x_val = df_val
    y_val = (df_val_label["category"] != normal_label).astype(int)
    # y_test: 1 = anomaly, 0 = normal
    y_test = (df_test_label["category"] != normal_label).astype(int)
    x_test = df_test

    # prepare datasets
    x_train_transformed = select_feature_set(preprocessor.transform(x_train), feature_set)
    x_val_transformed = select_feature_set(preprocessor.transform(x_val), feature_set)
    x_test_transformed = select_feature_set(preprocessor.transform(x_test), feature_set)

    # create detector on TRAIN
    detector = train_detector(x_train_transformed)

    # choose the F1 threshold on the mixed validation split
    val_scores = detector.score(x_val_transformed)
    threshold = choose_f1_threshold(y_val, val_scores)

    # evaluate on TEST
    anomaly_score = detector.score(x_test_transformed)

    return y_test, anomaly_score, threshold

def debug_csv(feature_set, normal_label, y_test, y_pred, anomaly_score):
    """
    Write the intermediate data to CSV file.
    """
    feature_set_name = "_".join(feature_set)
    class_label_name = normal_label

    output_path = f"tmp/if_{class_label_name}_{feature_set_name}.csv"
    pd.DataFrame({
        "y_test": y_test,
        "y_pred": y_pred,
        "anomaly_score": anomaly_score,
    }).to_csv(output_path, index=False)

def compute_scores(feature_set, normal_label):
    y_test, anomaly_score, threshold = evaluate(feature_set, normal_label)
    auc = roc_auc_score(y_test, anomaly_score)
    ap = average_precision_score(y_test, anomaly_score)
    y_pred = (anomaly_score >= threshold).astype(int)
    f1 = f1_score(y_test, y_pred)

    debug_csv(feature_set, normal_label, y_test, y_pred, anomaly_score)
    return {"auc_score": auc, "ap_score": ap, "f1_score": f1}

def plot_curves(feature_set, normal_label):
    y_test, anomaly_score, threshold = evaluate(feature_set, normal_label)

    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    PrecisionRecallDisplay.from_predictions(
        y_test,
        anomaly_score,
        name="IsolationForest PR-AUC",
        ax=axes[0],
    )
    axes[0].set_title(f"{normal_label} Precision-Recall")

    RocCurveDisplay.from_predictions(
        y_test,
        anomaly_score,
        name="IsolationForest AUC-ROC",
        ax=axes[1],
    )
    axes[1].set_title(f"{normal_label} ROC Curve")

    plt.tight_layout()
    plt.show()


## Evaluation

For each label listed in `category_labels`, that label is treated as the in-profile or normal class. All remaining labels are treated as anomalies.

The evaluation uses three disjoint splits:

- `train`: only samples from the selected normal class are used to fit the detector
- `val`: a mixed split is used to choose the anomaly-score threshold that maximizes F1
- `test`: the final metrics are reported using the threshold selected on validation

Each row in the result table corresponds to one target class and one feature-group combination.


In [6]:
from itertools import combinations

rows = []

group_names = list(feature_groups)
for size in range(1, len(group_names) + 1):
    for group_combo in combinations(group_names, size):
        selected_features = []
        for group_name in group_combo:
            selected_features.extend(feature_groups[group_name])

        feature_set_name = ", ".join(group_combo)

        for label in category_labels:
            display(f"Scoring {label}@{selected_features}...")
            scores = compute_scores(selected_features, label)

            rows.append({
                "FeatureSet": feature_set_name,
                "ClassLabel": label,
                "AucScore": scores["auc_score"],
                "ApScore": scores["ap_score"],
                "F1Score": scores["f1_score"],
            })

results_df = pd.DataFrame(rows, columns=["FeatureSet", "ClassLabel", "AucScore", "ApScore", "F1Score"])
display(results_df)


"Scoring system@['bs', 'ps', 'br', 'pr', 'td']..."

"Scoring malware@['bs', 'ps', 'br', 'pr', 'td']..."

"Scoring application@['bs', 'ps', 'br', 'pr', 'td']..."

"Scoring system@['tls.cver', 'tls.ccs', 'tls.cext', 'tls.csg', 'tls.alpn', 'tls.csv']..."

"Scoring malware@['tls.cver', 'tls.ccs', 'tls.cext', 'tls.csg', 'tls.alpn', 'tls.csv']..."

...

"Scoring application@['tls.cver', 'tls.ccs', 'tls.cext', 'tls.csg', 'tls.alpn', 'tls.csv', 'tls.sver', 'tls.scs', 'tls.sext', 'tls.ssv', 'tls.rec']..."

"Scoring system@['bs', 'ps', 'br', 'pr', 'td', 'tls.cver', 'tls.ccs', 'tls.cext', 'tls.csg', 'tls.alpn', 'tls.csv', 'tls.sver', 'tls.scs', 'tls.sext', 'tls.ssv', 'tls.rec']..."

"Scoring malware@['bs', 'ps', 'br', 'pr', 'td', 'tls.cver', 'tls.ccs', 'tls.cext', 'tls.csg', 'tls.alpn', 'tls.csv', 'tls.sver', 'tls.scs', 'tls.sext', 'tls.ssv', 'tls.rec']..."

"Scoring application@['bs', 'ps', 'br', 'pr', 'td', 'tls.cver', 'tls.ccs', 'tls.cext', 'tls.csg', 'tls.alpn', 'tls.csv', 'tls.sver', 'tls.scs', 'tls.sext', 'tls.ssv', 'tls.rec']..."

,FeatureSet,ClassLabel,AucScore,ApScore,F1Score
0,FLOW,system,0.865624,0.872959,0.862978
1,FLOW,malware,0.563709,0.581315,0.710667
2,FLOW,application,0.393270,0.981822,0.994388
3,CTLS,system,0.917965,0.935137,0.900189
4,CTLS,malware,0.267699,0.508121,0.736670
5,CTLS,application,0.683561,0.991929,0.994414
6,STLS,system,0.943530,0.940660,0.944621
7,STLS,malware,0.802230,0.804772,0.738667
8,STLS,application,0.714467,0.993917,0.994414
9,REC,system,0.781041,0.841049,0.802245


## System Profile

The table below reports the Isolation Forest baseline for the `system` profile across all feature-group combinations. Strong results here indicate that system traffic can be isolated from the rest of the dataset with relatively few random partitioning steps.


In [7]:
system_df = results_df[results_df["ClassLabel"] == "system"].sort_values("F1Score", ascending=False)
display(system_df)

,FeatureSet,ClassLabel,AucScore,ApScore,F1Score
15,"FLOW, STLS",system,0.934360,0.924308,0.947776
6,STLS,system,0.943530,0.940660,0.944621
42,"FLOW, CTLS, STLS, REC",system,0.949611,0.955133,0.943201
30,"FLOW, CTLS, STLS",system,0.954915,0.962147,0.938515
33,"FLOW, CTLS, REC",system,0.938405,0.944076,0.937836
21,"CTLS, STLS",system,0.953614,0.963213,0.937058
39,"CTLS, STLS, REC",system,0.945621,0.953211,0.934792
24,"CTLS, REC",system,0.947315,0.956400,0.933667
12,"FLOW, CTLS",system,0.939173,0.956195,0.931266
27,"STLS, REC",system,0.930685,0.930430,0.930266


## Malware Profile

This section isolates the Isolation Forest results for the `malware` profile. Comparing these rows with the `system` and `application` views highlights whether malware traffic remains separable when the detector relies on sparse tree partitions rather than density or reconstruction assumptions.


In [8]:
malware_df = results_df[results_df["ClassLabel"] == "malware"].sort_values("F1Score", ascending=False)
display(malware_df)

,FeatureSet,ClassLabel,AucScore,ApScore,F1Score
34,"FLOW, CTLS, REC",malware,0.563038,0.533531,0.746930
25,"CTLS, REC",malware,0.555520,0.533526,0.744014
7,STLS,malware,0.802230,0.804772,0.738667
19,"FLOW, REC",malware,0.705267,0.704754,0.737659
40,"CTLS, STLS, REC",malware,0.652696,0.594425,0.737446
43,"FLOW, CTLS, STLS, REC",malware,0.683879,0.608182,0.737342
4,CTLS,malware,0.267699,0.508121,0.736670
13,"FLOW, CTLS",malware,0.404892,0.455077,0.735904
10,REC,malware,0.734788,0.744713,0.733395
28,"STLS, REC",malware,0.877685,0.854321,0.731060


## Application Profile

This section isolates the Isolation Forest results for the `application` profile. Comparing it with the `system` and `malware` views shows whether application traffic forms a stable enough region of the feature space for tree-based isolation to separate it from the rest.


In [9]:
application_df = results_df[results_df["ClassLabel"] == "application"].sort_values("F1Score", ascending=False)
display(application_df)


,FeatureSet,ClassLabel,AucScore,ApScore,F1Score
44,"FLOW, CTLS, STLS, REC",application,0.725375,0.990310,0.994416
8,STLS,application,0.714467,0.993917,0.994414
5,CTLS,application,0.683561,0.991929,0.994414
17,"FLOW, STLS",application,0.654060,0.991891,0.994414
32,"FLOW, CTLS, STLS",application,0.691998,0.988917,0.994414
23,"CTLS, STLS",application,0.704706,0.991746,0.994414
14,"FLOW, CTLS",application,0.680677,0.988924,0.994414
41,"CTLS, STLS, REC",application,0.727740,0.990686,0.994414
35,"FLOW, CTLS, REC",application,0.704437,0.989700,0.994414
26,"CTLS, REC",application,0.714416,0.990892,0.994414
